In [1]:
import torch
import torch.nn as nn
import numpy as np
from typing import Dict, Tuple, List, Optional
from pathlib import Path
import cv2
from collections import Counter
import os


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

from tqdm import tqdm
from PIL import Image
from torchvision import transforms
import warnings
warnings.filterwarnings('ignore')

In [2]:
class UnifiedInferenceSystem:
    """
    Unified inference system for coronary artery dominance classification
    with occlusion detection and frame quality assessment.
    Handles both RCAMultiTaskStudent (with LSTM) and LCAMultiTaskModel.
    """
    
    def __init__(
        self,
        rca_model: nn.Module,
        lca_model: nn.Module,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
        occlusion_threshold: float = 0.5,
        quality_threshold: float = 0.5,
        dominance_threshold: float = 0.5,
        image_size: Tuple[int, int] = (512, 512),
        normalize_mean: List[float] = [0.485, 0.456, 0.406],
        normalize_std: List[float] = [0.229, 0.224, 0.225],
        occlusion_sequence_length: int = 8,  # Number of frames for LSTM
        occlusion_stride: int = 4  # Stride for sampling frames
    ):
        """
        Initialize the inference system.
        
        Args:
            rca_model: RCA MTL model (RCAMultiTaskStudent instance with LSTM)
            lca_model: LCA MTL model (LCAMultiTaskModel instance)
            device: Device to run inference on
            occlusion_threshold: Threshold for occlusion detection
            quality_threshold: Threshold for frame quality assessment
            image_size: Size to resize images to (H, W)
            normalize_mean: Mean values for normalization
            normalize_std: Std values for normalization
            occlusion_sequence_length: Number of frames to use for LSTM-based occlusion detection
            occlusion_stride: Stride for sampling frames in sequence
        """
        self.device = torch.device(device)
        self.occlusion_threshold = occlusion_threshold
        self.quality_threshold = quality_threshold
        self.dominance_threshold = dominance_threshold
        self.image_size = image_size
        self.normalize_mean = np.array(normalize_mean)
        self.normalize_std = np.array(normalize_std)
        self.occlusion_sequence_length = occlusion_sequence_length
        self.occlusion_stride = occlusion_stride
        
        # Set models to evaluation mode
        self.rca_model = rca_model.to(self.device)
        self.rca_model.eval()
        
        self.lca_model = lca_model.to(self.device)
        self.lca_model.eval()
        
        print(f"Models loaded successfully on {self.device}")
        print(f"Occlusion detection will use sequences of {occlusion_sequence_length} frames")
    
    @staticmethod
    def load_rca_model_from_checkpoint(
        checkpoint_path: str,
        model_class,
        backbone_fn,
        backbone_type: str,
        input_channels: int = 1,
        num_classes_occlusion: int = 2,
        num_classes_frame_quality: int = 2,
        num_classes_dominance: int = 2,
        hidden_dim: int = 256,
        sequential_model_type: str = 'lstm',
        sequential_hidden_dim: int = 256,
        num_sequential_layers: int = 1,
        bidirectional: bool = False,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ) -> nn.Module:
        """Helper method to load RCA model from checkpoint."""
        #backbone = backbone_fn(pretrained='DEFAULT')
        backbone = backbone_fn
        
        model = model_class(
            backbone=backbone,
            backbone_type=backbone_type,
            input_channels=input_channels,
            num_classes_occlusion=num_classes_occlusion,
            num_classes_frame_quality=num_classes_frame_quality,
            num_classes_dominance=num_classes_dominance,
            hidden_dim=hidden_dim,
            sequential_model_type=sequential_model_type,
            sequential_hidden_dim=sequential_hidden_dim,
            num_sequential_layers=num_sequential_layers,
            bidirectional=bidirectional
        )
        
        checkpoint = torch.load(checkpoint_path, map_location=device,weights_only=False)
        
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        elif 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        else:
            model.load_state_dict(checkpoint)
        
        model = model.to(device)
        model.eval()
        
        print(f"RCA model loaded from {checkpoint_path}")
        return model
    
    @staticmethod
    def load_lca_model_from_checkpoint(
        checkpoint_path: str,
        model_class,
        backbone_fn,
        backbone_type: str,
        input_channels: int = 3,
        num_classes_occlusion: int = 2,
        num_classes_frame_quality: int = 2,
        num_classes_dominance: int = 2,
        hidden_dim: int = 256,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ) -> nn.Module:
        """Helper method to load LCA model from checkpoint."""
        #backbone = backbone_fn(pretrained='DEFAULT')
        backbone = backbone_fn
        
        model = model_class(
            backbone=backbone,
            backbone_type=backbone_type,
            input_channels=input_channels,
            num_classes_occlusion=num_classes_occlusion,
            num_classes_frame_quality=num_classes_frame_quality,
            num_classes_dominance=num_classes_dominance,
            hidden_dim=hidden_dim
        )
        
        checkpoint = torch.load(checkpoint_path, map_location=device,weights_only=False)
        
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        elif 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        else:
            model.load_state_dict(checkpoint)
        
        model = model.to(device)
        model.eval()
        
        print(f"LCA model loaded from {checkpoint_path}")
        return model
    
    def load_frames_from_folder(
        self, 
        folder_path: str, 
        sort_frames: bool = True,
        grayscale: bool = True
    ) -> torch.Tensor:
        """
        Load all frames from a folder.
        
        Args:
            folder_path: Path to folder containing frame images
            sort_frames: Whether to sort frames by filename
            grayscale: Whether to load as grayscale (for RCA model)
            
        Returns:
            Tensor of shape (T, C, H, W) where T is number of frames
        """
        folder = Path(folder_path)
        
        if not folder.exists():
            raise ValueError(f"Folder does not exist: {folder_path}")
        
        # Get all image files
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
        image_files = [
            f for f in folder.iterdir() 
            if f.suffix.lower() in image_extensions
        ]
        
        if len(image_files) == 0:
            raise ValueError(f"No image files found in {folder_path}")
        
        # Sort frames by filename if requested
        if sort_frames:
            image_files = sorted(image_files, key=lambda x: x.name)
        
        frames = []
        for img_path in image_files:
            # Read image
            if grayscale:
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = np.expand_dims(img, axis=-1)  # Add channel dimension
            else:
                img = cv2.imread(str(img_path))
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            if img is None:
                print(f"Warning: Could not read {img_path}, skipping...")
                continue
            
            # Resize to model input size
            img = cv2.resize(img, self.image_size)
            
            # Normalize
            img = img.astype(np.float32) / 255.0
            
            if grayscale:
                # For grayscale, normalize with single channel
                img = (img - self.normalize_mean[0]) / self.normalize_std[0]  # Common grayscale normalization
            else:
                img = (img - self.normalize_mean) / self.normalize_std
            
            # Convert to CHW format
            #img = np.transpose(img, (2, 0, 1))
            img = np.expand_dims(img, axis=0)
            frames.append(img)
        
        if len(frames) == 0:
            raise ValueError(f"No valid frames could be loaded from {folder_path}")
        
        # Convert to tensor
        frames_tensor = torch.from_numpy(np.stack(frames, axis=0)).float()
        return frames_tensor
    
    def create_temporal_sequences(
        self, 
        frames: torch.Tensor, 
        seq_length: int = None, 
        stride: int = None
    ) -> torch.Tensor:
        """
        Create overlapping temporal sequences from frames for LSTM processing.
        
        Args:
            frames: Tensor of shape (T, C, H, W)
            seq_length: Length of each sequence
            stride: Stride between sequences
            
        Returns:
            Tensor of shape (N, seq_length, C, H, W) where N is number of sequences
        """
        if seq_length is None:
            seq_length = self.occlusion_sequence_length
        if stride is None:
            stride = self.occlusion_stride
        
        T = len(frames)
        
        # If we don't have enough frames, repeat the last frame
        if T < seq_length:
            padding_needed = seq_length - T
            last_frame = frames[-1:].repeat(padding_needed, 1, 1, 1)
            frames = torch.cat([frames, last_frame], dim=0)
            T = len(frames)
        
        # Create sequences
        sequences = []
        for i in range(0, T - seq_length + 1, stride):
            seq = frames[i:i + seq_length]
            sequences.append(seq)
        
        # If no sequences were created (shouldn't happen after padding), use all frames
        if len(sequences) == 0:
            sequences = [frames[:seq_length]]
        
        return torch.stack(sequences, dim=0)
    
    def detect_occlusion(self, frames: torch.Tensor) -> Tuple[bool, float]:
        #frames = frames.to(self.device)
        frames = frames.to(self.device).float()
        
        with torch.no_grad():
            # Use stride = sequence_length for NO overlap
            sequences = self.create_temporal_sequences(
                frames, 
                seq_length=self.occlusion_sequence_length,
                stride=self.occlusion_sequence_length  # No overlap
            )
            
            batch_size = 4
            all_logits = []
            
            for i in range(0, len(sequences), batch_size):
                batch_sequences = sequences[i:i+batch_size]
                
                input_dict = {'occlusion_images': batch_sequences}
                outputs = self.rca_model(input_dict)
                
                if 'occlusion' in outputs:
                    occlusion_logits = outputs['occlusion'].squeeze(1)
                    all_logits.append(occlusion_logits)
            
            if all_logits:
                all_logits = torch.cat(all_logits, dim=0)
                mean_logits = all_logits.mean(dim=0)
                mean_probs = torch.softmax(mean_logits, dim=-1)
                confidence = mean_probs[1].item()
            else:
                confidence = 0.0
            
            is_occluded = confidence > self.occlusion_threshold
        
        return is_occluded, confidence
    
    def assess_frame_quality(
        self, 
        frames: torch.Tensor, 
        model: nn.Module,
        is_rca: bool = False
    ) -> Tuple[torch.Tensor, List[int]]:
        """
        Assess quality of frames and filter out non-informative ones.
        
        Args:
            frames: Video frames tensor (T, C, H, W)
            model: Model to use (RCA or LCA)
            is_rca: Whether using RCA model (requires different input format)
            
        Returns:
            (informative_frames, informative_indices)
        """
        #frames = frames.to(self.device)
        frames = frames.to(self.device).float()
        
        with torch.no_grad():
            # Process frames in batches
            batch_size = 16
            quality_scores = []
            
            for i in range(0, len(frames), batch_size):
                batch = frames[i:i+batch_size]
                
                if is_rca:
                    # RCA model expects dict input
                    input_dict = {'frame_quality_images': batch}
                    outputs = model(input_dict)
                else:
                    # LCA model expects direct tensor input
                    outputs = model(batch)
                
                if 'frame_quality' in outputs:
                    quality_logits = outputs['frame_quality'].squeeze(1)  # Remove temporal dim if present
                    
                    # Apply softmax for binary classification
                    if quality_logits.shape[-1] == 2:
                        quality_probs = torch.softmax(quality_logits, dim=-1)
                        # Probability of class 1 (informative/good quality)
                        quality_prob = quality_probs[:, 1]
                    else:
                        # Single output with sigmoid
                        quality_prob = torch.sigmoid(quality_logits).squeeze(-1)
                    
                    quality_scores.append(quality_prob)
            
            # Concatenate all scores
            if quality_scores:
                quality_scores = torch.cat(quality_scores, dim=0)
                
                # Filter informative frames
                informative_mask = quality_scores > self.quality_threshold
                informative_indices = torch.where(informative_mask)[0].cpu().tolist()
                informative_frames = frames[informative_mask]
            else:
                informative_frames = frames
                informative_indices = list(range(len(frames)))
        
        if len(informative_frames) == 0:
            print("Warning: No informative frames found. Using all frames.")
            return frames, list(range(len(frames)))
        
        return informative_frames, informative_indices
    
    def classify_dominance(
        self, 
        frames: torch.Tensor, 
        model: nn.Module,
        is_rca: bool = False,
    ) -> Tuple[str, float, List[str], List[float]]:
        """
        Classify coronary dominance for each frame and perform majority voting.
        
        Args:
            frames: Informative frames tensor (T, C, H, W)
            model: Model to use (RCA or LCA)
            is_rca: Whether using RCA model (requires different input format)
            
        Returns:
            (final_decision, confidence, frame_predictions, frame_confidences)
        """
        #frames = frames.to(self.device)
        frames = frames.to(self.device).float()
        
        # Class names - adjust if your model uses different class ordering
        #class_names = ['left', 'right']  # Adjust based on your training
        class_names = ['rightdom', 'leftdom']
        
        with torch.no_grad():
            # Process frames in batches
            batch_size = 16
            all_predictions = []
            all_confidences = []
            
            for i in range(0, len(frames), batch_size):
                batch = frames[i:i+batch_size]
                
                if is_rca:
                    # RCA model expects dict input
                    input_dict = {'dominance_images': batch}
                    outputs = model(input_dict)
                else:
                    # LCA model expects direct tensor input
                    outputs = model(batch)
                
                if 'dominance' in outputs:
                    dominance_logits = outputs['dominance'].squeeze(1)  # Remove temporal dim if present
                    
                    # Apply softmax for multi-class classification
                    dominance_probs = torch.softmax(dominance_logits, dim=-1)
                    
                    # Get predictions and confidences
                    max_probs, preds = torch.max(dominance_probs, dim=-1)
                    
                    all_predictions.append(preds)
                    all_confidences.append(max_probs)
            
            # Concatenate all predictions
            if all_predictions:
                all_predictions = torch.cat(all_predictions, dim=0).cpu().numpy()
                all_confidences = torch.cat(all_confidences, dim=0).cpu().numpy()
            else:
                all_predictions = np.array([])
                all_confidences = np.array([])
        
        if len(all_predictions) == 0:
            return 'unknown', 0.0, [], []
        
        # Convert to class names
        frame_predictions = [class_names[pred] for pred in all_predictions]
        frame_confidences = all_confidences.tolist()

        # FILTER: Only keep predictions with high confidence
        confident_votes = [
            pred for pred, conf in zip(frame_predictions, frame_confidences)
            if conf > self.dominance_threshold
        ]
        
        if len(confident_votes) == 0:
            print(f"Warning: No frames exceed confidence threshold ({self.dominance_threshold})")
            print(f"Max confidence found: {max(frame_confidences):.4f}")
            # Fallback: use all predictions
            confident_votes = frame_predictions

        # Majority voting
        vote_counts = Counter(frame_predictions)
        final_decision = vote_counts.most_common(1)[0][0]
        
        # Calculate confidence as mean confidence of frames that voted for final decision
        decision_confidences = [
            conf for pred, conf in zip(frame_predictions, frame_confidences) 
            if pred == final_decision
        ]
        final_confidence = float(np.mean(decision_confidences)) if decision_confidences else 0.0
        
        return final_decision, final_confidence, frame_predictions, frame_confidences
    
    def inference(
        self, 
        rca_folder_path: str, 
        lca_folder_path: str,
        verbose: bool = True
    ) -> Dict:
        """
        Perform complete inference on RCA and LCA frame folders.
        
        Args:
            rca_folder_path: Path to folder containing RCA frames
            lca_folder_path: Path to folder containing LCA frames
            verbose: Whether to print detailed information
            
        Returns:
            Dictionary containing inference results
        """
        results = {
            'rca_occluded': False,
            'video_used': 'RCA',
            'num_total_frames': 0,
            'num_informative_frames': 0,
            'informative_frame_indices': [],
            'frame_predictions': [],
            'frame_confidences': [],
            'final_decision': '',
            'confidence': 0.0,
            'occlusion_confidence': 0.0
        }
        
        # Step 1 & 2: Load RCA frames and check occlusion
        if verbose:
            print("\n" + "="*60)
            print("STEP 1: Loading RCA Frames")
            print("="*60)
        
        # RCA model uses grayscale
        rca_frames = self.load_frames_from_folder(rca_folder_path, grayscale=True)
        results['num_total_frames'] = len(rca_frames)
        
        if verbose:
            print(f"Loaded {len(rca_frames)} grayscale frames from {rca_folder_path}")
            print("\n" + "="*60)
            print("STEP 2: Occlusion Detection (LSTM-based)")
            print("="*60)
        
        is_occluded, occlusion_conf = self.detect_occlusion(rca_frames)
        results['rca_occluded'] = is_occluded
        results['occlusion_confidence'] = occlusion_conf
        
        if verbose:
            print(f"RCA Occlusion Status: {'OCCLUDED' if is_occluded else 'NOT OCCLUDED'}")
            print(f"Occlusion Confidence: {occlusion_conf:.4f}")
        
        # Decide which video to use
        if is_occluded:
            if verbose:
                print("\n" + "="*60)
                print("RCA is occluded. Switching to LCA frames...")
                print("="*60)
            
            results['video_used'] = 'LCA'
            # LCA model uses RGB
            frames = self.load_frames_from_folder(lca_folder_path, grayscale=True)
            model = self.lca_model
            is_rca = False
            
            if verbose:
                print(f"Loaded {len(frames)} RGB frames from {lca_folder_path}")
        else:
            if verbose:
                print("\nProceeding with RCA frames...")
            
            results['video_used'] = 'RCA'
            frames = rca_frames
            model = self.rca_model
            is_rca = True
        
        # Step 3: Frame quality assessment
        if verbose:
            print("\n" + "="*60)
            print("STEP 3: Frame Quality Assessment")
            print("="*60)
        
        informative_frames, informative_indices = self.assess_frame_quality(
            frames, model, is_rca=is_rca
        )
        results['num_informative_frames'] = len(informative_frames)
        results['informative_frame_indices'] = informative_indices
        
        if verbose:
            print(f"Informative Frames: {len(informative_frames)}/{len(frames)}")
            print(f"Filtered out: {len(frames) - len(informative_frames)} non-informative frames")
        
        # Steps 4 & 5: Dominance classification with majority voting
        if verbose:
            print("\n" + "="*60)
            print("STEP 4 & 5: Dominance Classification & Majority Voting")
            print("="*60)
        
        final_decision, confidence, frame_preds, frame_confs = self.classify_dominance(
            informative_frames, model, is_rca=is_rca
        )
        results['final_decision'] = final_decision
        results['confidence'] = confidence
        results['frame_predictions'] = frame_preds
        results['frame_confidences'] = frame_confs
        
        if verbose:
            print(f"\nFrame-wise predictions:")
            pred_counts = Counter(frame_preds)
            for pred_class, count in pred_counts.most_common():
                percentage = (count / len(frame_preds)) * 100 if frame_preds else 0
                print(f"  {pred_class}: {count}/{len(frame_preds)} ({percentage:.1f}%)")
            
            print("\n" + "="*60)
            print("FINAL RESULT")
            print("="*60)
            print(f"Video Used: {results['video_used']}")
            print(f"Coronary Dominance: {final_decision.upper()}")
            print(f"Confidence: {confidence:.4f}")
            print("="*60 + "\n")
        
        return results
    
    def batch_inference(
        self, 
        folder_pairs: List[Tuple[str, str]], 
        output_file: Optional[str] = None
    ) -> List[Dict]:
        """
        Perform inference on multiple folder pairs.
        
        Args:
            folder_pairs: List of (rca_folder_path, lca_folder_path) tuples
            output_file: Optional path to save results
            
        Returns:
            List of result dictionaries
        """
        all_results = []
        
        for idx, (rca_folder, lca_folder) in enumerate(folder_pairs):
            path_str = str(rca_folder)
            path_parts = Path(path_str).parts
            study_id = path_parts[7].split('_')[0]
            
            print(f"\n{'='*60}")
            #print(f"Processing folder pair {idx + 1}/{len(folder_pairs)}")
            print(f"Processing folder pair {study_id}/{len(folder_pairs)}")
            print(f"{'='*60}")
            
            try:
                results = self.inference(rca_folder, lca_folder, verbose=True)
                results['pair_id'] = study_id
                results['rca_folder'] = rca_folder
                results['lca_folder'] = lca_folder
                all_results.append(results)
            except Exception as e:
                #print(f"Error processing folder pair {idx + 1}: {str(e)}")
                print(f"Error processing folder pair {study_id}: {str(e)}")
                import traceback
                traceback.print_exc()
                continue
        
        # Save results if output file specified
        if output_file:
            import json
            with open(output_file, 'w') as f:
                json.dump(all_results, f, indent=2)
            print(f"\nResults saved to {output_file}")
        
        # Print summary statistics
        if all_results:
            print("\n" + "="*60)
            print("BATCH INFERENCE SUMMARY")
            print("="*60)
            print(f"Total pairs processed: {len(all_results)}")
            
            decisions = [r['final_decision'] for r in all_results]
            decision_counts = Counter(decisions)
            print("\nFinal Decisions Distribution:")
            for decision, count in decision_counts.most_common():
                percentage = (count / len(all_results)) * 100
                print(f"  {decision}: {count}/{len(all_results)} ({percentage:.1f}%)")
            
            occlusion_count = sum(1 for r in all_results if r['rca_occluded'])
            print(f"\nRCA Occlusions: {occlusion_count}/{len(all_results)} ({occlusion_count/len(all_results)*100:.1f}%)")
            
            avg_confidence = np.mean([r['confidence'] for r in all_results])
            print(f"Average Confidence: {avg_confidence:.4f}")
            print("="*60 + "\n")
        
        return all_results

# BEGIN INFERENCE

In [3]:
os.environ['TORCH_HOME'] = r'E:\MS Thesis\codes\DSTCT-master\code\weights'

In [4]:
import os
print(os.getcwd())
os.chdir(r"E:\Morshedul\CoronarDominance\INFERENCE_SYSTEM")
print(os.getcwd())

E:\Morshedul\CoronarDominance
E:\Morshedul\CoronarDominance\INFERENCE_SYSTEM


In [5]:
from torchvision import models
from mtl_models import RCAMultiTaskStudent, LCAMultiTaskModel

In [6]:
categories_map = {
    'occlusion': ['nonoccluded', 'occluded'],
    'frame_quality': ['noninformative', 'informative'],
    'dominance': ['rightdom', 'leftdom']
}

In [27]:
#RCA_CHECKPOINT_PATH = r'E:\Morshedul\CoronarDominance\MTL_results\RCA\MTD_MTL_Resnet18_LSTM\training_results\TemporalMTD_Resnet18_fold1_best.pth'
#LCA_CHECKPOINT_PATH = r'E:\Morshedul\CoronarDominance\MTL_results\LCA\MTD_Resnet18_LSTM\training_results\Resnet18_mtd_mtl_lstm_best_LCA_best.pth'
#BACKBONE_FN = models.resnet18(weights="DEFAULT")
#BACKBONE_TYPE = 'resnet18'

#RCA_CHECKPOINT_PATH = r'E:\Morshedul\CoronarDominance\MTL_results\RCA\MTD_MTL_MobileNetV2_LSTM\training_results\TemporalMTD_MobileNetV2_fold1_best.pth'
#LCA_CHECKPOINT_PATH = r'E:\Morshedul\CoronarDominance\MTL_results\LCA\MTD_MobileNetV2_LSTM\training_results\MobileNetV2_mtd_mtl_lstm_best_LCA_best.pth'
#BACKBONE_FN = models.mobilenet_v2(weights="DEFAULT")
#BACKBONE_TYPE = 'mobilenet_v2'

RCA_CHECKPOINT_PATH = r'E:\Morshedul\CoronarDominance\MTL_results\RCA\MTD_MTL_Densenet121_LSTM\training_results\TemporalMTD_Densenet121_fold1_best.pth'
LCA_CHECKPOINT_PATH = r'E:\Morshedul\CoronarDominance\MTL_results\LCA\MTD_Densenet121_LSTM\training_results\MobileNetV2_mtd_mtl_lstm_best_LCA_best.pth'
BACKBONE_FN = models.densenet121(weights="DEFAULT")
BACKBONE_TYPE = 'densenet121'


INPUT_CHANNELS = 1
HIDDEN_DIM = 128
SEQUENTIAL_HIDDEN_DIM = 128
NUM_SEQUENTIAL_LAYERS = 5

IMAGE_SIZE = (512, 512)
OCCLUSION_THRESHOLD = 0.5
QUALITY_THRESHOLD = 0.5
DOMINANCE_THRESHOLD = 0.5
OCCLUSION_SEQUENCE_LENGTH = 8
OCCLUSION_STRIDE = 4
NORMALIZE_MEAN = [0.5485]
NORMALIZE_STD = [0.1407]

In [28]:
# Load RCA model (with LSTM)
rca_model = UnifiedInferenceSystem.load_rca_model_from_checkpoint(
    checkpoint_path=RCA_CHECKPOINT_PATH,
    model_class=RCAMultiTaskStudent,
    backbone_fn=BACKBONE_FN,
    backbone_type=BACKBONE_TYPE,
    input_channels=INPUT_CHANNELS,  # Grayscale
    num_classes_occlusion=2,
    num_classes_frame_quality=2,
    num_classes_dominance=2,
    hidden_dim=HIDDEN_DIM,
    sequential_model_type='lstm',
    sequential_hidden_dim=SEQUENTIAL_HIDDEN_DIM,
    num_sequential_layers=NUM_SEQUENTIAL_LAYERS,
    bidirectional=False
)

RCA model loaded from E:\Morshedul\CoronarDominance\MTL_results\RCA\MTD_MTL_Densenet121_LSTM\training_results\TemporalMTD_Densenet121_fold1_best.pth


In [29]:
# Load LCA model
lca_model = UnifiedInferenceSystem.load_lca_model_from_checkpoint(
    checkpoint_path=LCA_CHECKPOINT_PATH,
    model_class=LCAMultiTaskModel,
    backbone_fn=BACKBONE_FN,
    backbone_type=BACKBONE_TYPE,
    input_channels=INPUT_CHANNELS,  # RGB
    num_classes_occlusion=2,
    num_classes_frame_quality=2,
    num_classes_dominance=2,
    hidden_dim=HIDDEN_DIM
)

DenseNet conv0 modified to 1 input channel(s).
LCA model loaded from E:\Morshedul\CoronarDominance\MTL_results\LCA\MTD_Densenet121_LSTM\training_results\MobileNetV2_mtd_mtl_lstm_best_LCA_best.pth


In [30]:
# Initialize inference system
inference_system = UnifiedInferenceSystem(
    rca_model=rca_model,
    lca_model=lca_model,
    device='cuda',
    occlusion_threshold=OCCLUSION_THRESHOLD,
    quality_threshold=QUALITY_THRESHOLD,
    dominance_threshold = DOMINANCE_THRESHOLD,
    image_size=IMAGE_SIZE,
    normalize_mean=NORMALIZE_MEAN,
    normalize_std=NORMALIZE_STD,
    occlusion_sequence_length=OCCLUSION_SEQUENCE_LENGTH,  # Number of frames for LSTM
    occlusion_stride=OCCLUSION_STRIDE  # Stride for sampling
)

Models loaded successfully on cuda
Occlusion detection will use sequences of 8 frames


In [11]:
from pathlib import Path
import random
folder_pairs = []
num_studies = 20
random.seed(42)
right_dominance_root=r'E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance'
left_dominance_root=r'E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Left_Dominance'

for dominance_type, root_path in [('right', right_dominance_root),('left', left_dominance_root)]:
    root = Path(root_path)
    print(root)
    if not root.exists():
        print(f"Warning: {root_path} does not exist, skipping...")
        continue

     # Get all study folders sorted
    all_studies = sorted([d for d in root.iterdir() if d.is_dir()])
    
    # Select last N studies
    #selected_studies = all_studies[-num_studies:]
    #selected_studies = all_studies[:num_studies]
    selected_studies = all_studies[143:243]
    #selected_studies = all_studies.copy()
    
    print(f"\nProcessing {dominance_type} dominance: {len(selected_studies)} studies")
        

    for study_folder in selected_studies:
        print(study_folder)

        if not study_folder.is_dir():
            continue
        
        rca_dir = study_folder / 'RCA'
        lca_dir = study_folder / 'LCA'

        print(rca_dir)
        print(lca_dir)

        # Check if both RCA and LCA directories exist
        if not rca_dir.exists() or not lca_dir.exists():
            print(f"Warning: RCA or LCA missing for {study_folder.name}, skipping...")
            continue

         # Get all video folders in RCA (each video folder contains frames)
        rca_videos = [d for d in rca_dir.iterdir() if d.is_dir()]
        lca_videos = [d for d in lca_dir.iterdir() if d.is_dir()]
        print(len(rca_videos))
        print(len(lca_videos))

        if not rca_videos or not lca_videos:
            print(f"Warning: No video folders found for {study_folder.name}, skipping...")
            continue

        # Randomly select one video from each
        selected_rca = random.choice(rca_videos)
        selected_lca = random.choice(lca_videos)

        print(f"Selected RCA video: {selected_rca.name}")
        print(f"Selected LCA video: {selected_lca.name}")

        # Create paths to frames folders
        rca_frames_path = selected_rca / 'frames'
        lca_frames_path = selected_lca / 'frames'

         # Verify frames folders exist
        if not rca_frames_path.exists() or not lca_frames_path.exists():
            print(f"Warning: Frames folder missing for {study_folder.name}, skipping...")
            continue
        
        # Add to folder pairs
        folder_pairs.append((str(rca_frames_path), str(lca_frames_path)))
        
        print(f"✓ Added {study_folder.name}")
        
print(f"\nTotal folder pairs created: {len(folder_pairs)}")



E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance

Processing right dominance: 100 studies
E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance\Study00194_1.1.11.11111.11.891613822454460198705157870550996363
E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance\Study00194_1.1.11.11111.11.891613822454460198705157870550996363\RCA
E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance\Study00194_1.1.11.11111.11.891613822454460198705157870550996363\LCA
1
5
Selected RCA video: 1.1.11.111111.11.78612776651587438215644123011381083.1.1
Selected LCA video: 1.1.11.111111.11.18726912062804151889243514505487621.1.1
✓ Added Study00194_1.1.11.11111.11.891613822454460198705157870550996363
E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance\Study00195_1.1.11.11111.11.618492602933460103736020448084402169
E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance\Study00195_1.1.

In [12]:
len(folder_pairs)

200

In [31]:
batch_results = inference_system.batch_inference(
    folder_pairs=folder_pairs,
    output_file='Densenet121inference_results_testset.json'
)


Processing folder pair Study00194/200

STEP 1: Loading RCA Frames
Loaded 73 grayscale frames from E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance\Study00194_1.1.11.11111.11.891613822454460198705157870550996363\RCA\1.1.11.111111.11.78612776651587438215644123011381083.1.1\frames

STEP 2: Occlusion Detection (LSTM-based)
RCA Occlusion Status: OCCLUDED
Occlusion Confidence: 0.5700

RCA is occluded. Switching to LCA frames...
Loaded 37 RGB frames from E:\Morshedul\CoronarDominance\Extracted\main_normal\normal\Right_Dominance\Study00194_1.1.11.11111.11.891613822454460198705157870550996363\LCA\1.1.11.111111.11.18726912062804151889243514505487621.1.1\frames

STEP 3: Frame Quality Assessment
Informative Frames: 37/37
Filtered out: 0 non-informative frames

STEP 4 & 5: Dominance Classification & Majority Voting

Frame-wise predictions:
  rightdom: 37/37 (100.0%)

FINAL RESULT
Video Used: LCA
Coronary Dominance: RIGHTDOM
Confidence: 0.6950


Processing folder pair Stud

## INFERENCE EVALUATION

In [23]:
def extract_ground_truth_from_paths(folder_pairs: List[Tuple[str, str]]) -> Dict[str, str]:
    """
    Extract ground truth labels from folder paths.
    
    Assumes paths contain either 'Right_Dominant' or 'Left_Dominant'.
    Also extracts study ID from the path.
    
    Args:
        folder_pairs: List of (rca_path, lca_path) tuples
        
    Returns:
        Dictionary mapping study_id -> ground_truth_label
        e.g., {'study_1': 'rightdom', 'study_2': 'leftdom', ...}
    """
    ground_truth_data = {}
    
    for idx, (rca_path, lca_path) in enumerate(folder_pairs):
        # Use rca_path to extract ground truth
        path_str = str(rca_path)
        
        # Determine ground truth from path
        if 'Right_Dominance' in path_str:
            #print("rightdom")
            ground_truth = 'rightdom'
        elif 'Left_Dominance' in path_str:
            ground_truth = 'leftdom'
            #print("leftdom")
        else:
            raise ValueError(f"Could not determine ground truth from path: {path_str}")
        
        path_parts = Path(path_str).parts
        study_id = path_parts[7].split('_')[0]
        
        ground_truth_data[study_id] = ground_truth
    
    return ground_truth_data

In [24]:
ground_truth_data = extract_ground_truth_from_paths(folder_pairs)
print("Ground Truth Extracted:")
print(ground_truth_data)
print(len(ground_truth_data))

Ground Truth Extracted:
{'Study00194': 'rightdom', 'Study00195': 'rightdom', 'Study00196': 'rightdom', 'Study00197': 'rightdom', 'Study00198': 'rightdom', 'Study00199': 'rightdom', 'Study00200': 'rightdom', 'Study00201': 'rightdom', 'Study00202': 'rightdom', 'Study00203': 'rightdom', 'Study00204': 'rightdom', 'Study00205': 'rightdom', 'Study00206': 'rightdom', 'Study00207': 'rightdom', 'Study00208': 'rightdom', 'Study00209': 'rightdom', 'Study00210': 'rightdom', 'Study00211': 'rightdom', 'Study00212': 'rightdom', 'Study00213': 'rightdom', 'Study00214': 'rightdom', 'Study00215': 'rightdom', 'Study00216': 'rightdom', 'Study00217': 'rightdom', 'Study00218': 'rightdom', 'Study00219': 'rightdom', 'Study00220': 'rightdom', 'Study00221': 'rightdom', 'Study00222': 'rightdom', 'Study00223': 'rightdom', 'Study00224': 'rightdom', 'Study00225': 'rightdom', 'Study00226': 'rightdom', 'Study00227': 'rightdom', 'Study00228': 'rightdom', 'Study00229': 'rightdom', 'Study00230': 'rightdom', 'Study00231':

In [25]:
class InferenceEvaluator:
    """Evaluate inference results against ground truth (using dicts, not JSON files)."""
    
    def __init__(self, results: List[Dict], ground_truth: Dict[str, str]):
        """
        Args:
            results: List of result dictionaries from batch_inference
            ground_truth: Dictionary mapping study_id -> 'rightdom' or 'leftdom'
        """
        self.results = results
        self.ground_truth = ground_truth
    
    def evaluate(self) -> Dict:
        """Calculate all evaluation metrics."""
        
        # Extract predictions and true labels in matching order
        study_ids = []
        predictions = []
        true_labels = []
        confidences = []
        
        for result in self.results:
            study_id = result.get('pair_id', len(study_ids))
            
            if study_id not in self.ground_truth:
                print(f"Warning: {study_id} not in ground truth, skipping...")
                continue
            
            study_ids.append(study_id)
            predictions.append(result['final_decision'])
            true_labels.append(self.ground_truth[study_id])
            confidences.append(result.get('confidence', 0.0))
        
        # Convert to integers for sklearn
        label_to_int = {'rightdom': 0, 'leftdom': 1}
        y_pred = np.array([label_to_int[p] for p in predictions])
        y_true = np.array([label_to_int[t] for t in true_labels])
        confidences = np.array(confidences)
        
        metrics = {}
        
        # Basic metrics
        metrics['accuracy'] = float(accuracy_score(y_true, y_pred))
        metrics['f1_score'] = float(f1_score(y_true, y_pred, average='binary'))
        metrics['precision'] = float(precision_score(y_true, y_pred, average='binary'))
        metrics['recall'] = float(recall_score(y_true, y_pred, average='binary'))
        metrics['sensitivity'] = float(recall_score(y_true, y_pred, average='binary'))
        metrics['specificity'] = float(self._specificity(y_true, y_pred))
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        metrics['cm'] = cm.tolist()
        metrics['tn'] = int(cm[0, 0])
        metrics['fp'] = int(cm[0, 1])
        metrics['fn'] = int(cm[1, 0])
        metrics['tp'] = int(cm[1, 1])
        
        # Classification report
        metrics['classification_report'] = classification_report(
            y_true, y_pred,
            target_names=['rightdom', 'leftdom'],
            output_dict=True,
            digits=4
        )
        
        # Study-wise details
        metrics['study_details'] = [
            {
                'study_id': sid,
                'ground_truth': true_labels[i],
                'prediction': predictions[i],
                'confidence': float(confidences[i]),
                'correct': predictions[i] == true_labels[i]
            }
            for i, sid in enumerate(study_ids)
        ]
        
        return metrics
    
    @staticmethod
    def _specificity(y_true, y_pred):
        """Calculate specificity (true negative rate)."""
        cm = confusion_matrix(y_true, y_pred)
        tn = cm[0, 0]
        fp = cm[0, 1]
        return tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    def print_report(self, metrics: Dict):
        """Pretty print evaluation metrics."""
        print("\n" + "="*60)
        print("INFERENCE EVALUATION REPORT")
        print("="*60)
        
        print("\nOVERALL METRICS:")
        print(f"  Accuracy:    {metrics['accuracy']:.4f}")
        print(f"  F1-Score:    {metrics['f1_score']:.4f}")
        print(f"  Precision:   {metrics['precision']:.4f}")
        print(f"  Recall:      {metrics['recall']:.4f}")
        print(f"  Specificity: {metrics['specificity']:.4f}")
        
        print("\nCONFUSION MATRIX:")
        print(f"  True Negatives (TN):  {metrics['tn']}")
        print(f"  False Positives (FP): {metrics['fp']}")
        print(f"  False Negatives (FN): {metrics['fn']}")
        print(f"  True Positives (TP):  {metrics['tp']}")
        
        print("\nSTUDY-WISE RESULTS:")
        correct = 0
        for detail in metrics['study_details']:
            status = "✓" if detail['correct'] else "✗"
            print(f"  {status} {detail['study_id']}: "
                  f"True={detail['ground_truth']}, "
                  f"Pred={detail['prediction']}, "
                  f"Conf={detail['confidence']:.4f}")
            if detail['correct']:
                correct += 1
        
        print(f"\n  Correct: {correct}/{len(metrics['study_details'])}")
        print("="*60 + "\n")
    
    def save_results(self, metrics: Dict, output_path: str):
        """Save evaluation results to JSON."""
        import json
        with open(output_path, 'w') as f:
            json.dump(metrics, f, indent=2, default=str)
        print(f"Evaluation results saved to {output_path}")


In [32]:
# Step 3: Evaluate (assuming results from step 1)
evaluator = InferenceEvaluator(
    results=batch_results,
    ground_truth=ground_truth_data
)

metrics = evaluator.evaluate()
evaluator.print_report(metrics)
evaluator.save_results(metrics, 'Densenet121evaluation_metrics_testset.json')



INFERENCE EVALUATION REPORT

OVERALL METRICS:
  Accuracy:    0.9000
  F1-Score:    0.8889
  Precision:   1.0000
  Recall:      0.8000
  Specificity: 1.0000

CONFUSION MATRIX:
  True Negatives (TN):  100
  False Positives (FP): 0
  False Negatives (FN): 20
  True Positives (TP):  80

STUDY-WISE RESULTS:
  ✓ Study00194: True=rightdom, Pred=rightdom, Conf=0.6950
  ✓ Study00195: True=rightdom, Pred=rightdom, Conf=0.6912
  ✓ Study00196: True=rightdom, Pred=rightdom, Conf=0.6915
  ✓ Study00197: True=rightdom, Pred=rightdom, Conf=0.7395
  ✓ Study00198: True=rightdom, Pred=rightdom, Conf=0.8185
  ✓ Study00199: True=rightdom, Pred=rightdom, Conf=0.6955
  ✓ Study00200: True=rightdom, Pred=rightdom, Conf=0.8054
  ✓ Study00201: True=rightdom, Pred=rightdom, Conf=0.7805
  ✓ Study00202: True=rightdom, Pred=rightdom, Conf=0.7194
  ✓ Study00203: True=rightdom, Pred=rightdom, Conf=0.7715
  ✓ Study00204: True=rightdom, Pred=rightdom, Conf=0.7405
  ✓ Study00205: True=rightdom, Pred=rightdom, Conf=0.6807